# 02 — BI analýza: Nováčci na road vs. trail závodech
### 4IZ503 Projektový seminář — Ultra Marathon Running

---

### Výzkumná otázka

Jsou nováčci (první start) na trail závodech více penalizováni než na silničních?
Jinými slovy: ztrácí nezkušený závodník více času oproti veteránům na trailu,
kde záleží na orientaci, výbavě a technice — nebo na silnici, kde je úspěch
více otázkou pacing strategie a vytrvalosti?

**Hypotéza:** Trail penalizuje nezkušenost více než silnice. Důvody:
- orientace v terénu (značení, mapy)
- technická výbava (hole, batoh, nutrice)
- strategie výstupu/sestupu
- adaptace na nepředvídatelné podmínky (počasí, povrch)

**Business interpretace:** Pokud je hypotéza potvrzena, trail organizátoři by
měli nabízet kratší 'beginner' varianty nebo edukační workshopy pro nováčky —
jinak je velký podíl prvozávodníků odradí od opakované účasti.

---

### Účel notebooku

Tento notebook je součástí **BI části** projektu (ne DM/CleverMiner úloha).
Slouží jako podklad pro Power BI dashboard — vizualizuje vztah mezi zkušeností,
povrchem a rizikem zpomalení.

### Limitace

- Analýza pracuje pouze se závody, kde známe povrch (`surface IN ('road','trail')`)
- `speed_cat` je počítán **per event** (v notebooku 00) — srovnáváme relativní
  pozici ve startovním poli, ne absolutní rychlost
- `experience_cat` je odvozen z počtu startů v datasetu — nezahrnuje předchozí
  starty mimo ultramaratonský dataset (např. silniční maratony, trail tréninky)

## 1. Import a načtení dat

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/processed')
df = pd.read_parquet(DATA_DIR / 'ultra_clean.parquet')
print(f"Načteno: {len(df):,} řádků")
print(f"Sloupce: {df.columns.tolist()}")

## 2. Příprava dat

Filtrujeme záznamy s definovaným povrchem (road/trail) a kompletními údaji
o zkušenosti a rychlosti.

In [ ]:
# Filtr: road a trail závody + kompletní záznamy
df_filtered = df[
    df['surface'].isin(['trail', 'road']) &
    df['experience_cat'].notna() &
    df['speed_cat'].notna() &
    df['avg_speed'].notna()
].copy()

print(f"Záznamy s road/trail povrchem: {len(df_filtered):,}")
print()
print("Rozložení dle povrchu:")
print(df_filtered['surface'].value_counts())
print()
print("Rozložení dle zkušenosti:")
print(df_filtered['experience_cat'].value_counts())
print()
print("Ověření speed_cat per event (~33/33/33):")
print((df_filtered['speed_cat'].value_counts() / len(df_filtered) * 100).round(1))

## 3. Hlavní analýza: podíl pomalých závodníků dle zkušeností a povrchu

In [ ]:
# Logické pořadí (od nováčka k veteránovi)
exp_order = ['nováček', 'zkušený', 'veterán']
surface_order = ['road', 'trail']

# Pivot: podíl 'pomalý' (%) podle experience_cat × surface
pivot = df_filtered.groupby(['experience_cat', 'surface'])['speed_cat'].apply(
    lambda x: (x == 'pomalý').sum() / len(x) * 100
).unstack()
pivot = pivot.reindex(index=exp_order, columns=surface_order)

print("Podíl závodníků v kategorii 'pomalý' (%) dle zkušeností a povrchu:")
print(pivot.round(2))

# Počty pro transparentnost
counts = df_filtered.groupby(['experience_cat', 'surface']).size().unstack()
counts = counts.reindex(index=exp_order, columns=surface_order)
print("\nPočty závodníků v každé buňce:")
print(counts)

In [ ]:
# Relativní nevýhoda nováčků oproti veteránům (klíčová metrika)
rel_disadv = pivot.loc['nováček'] - pivot.loc['veterán']

print("Relativní nevýhoda nováčků oproti veteránům (procentní body):")
print("(kladné = nováčci mají o tolik pb vyšší podíl pomalých než veteráni)")
for surf in surface_order:
    print(f"  {surf.capitalize():6s}: +{rel_disadv[surf]:.2f} pb")

# Sanity check: průměrná rychlost dle skupiny
print("\nSanity check — průměrná rychlost (km/h):")
mean_speeds = df_filtered.groupby(['experience_cat', 'surface'])['avg_speed'].mean().unstack()
mean_speeds = mean_speeds.reindex(index=exp_order, columns=surface_order)
print(mean_speeds.round(2))
print("(očekávání: veteráni > zkušení > nováčci v každém povrchu)")

## 4. Vizualizace

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Výkonnost nováčků vs. veteránů dle povrchu závodu\n(road vs. trail, speed_cat per event)',
             fontsize=13, fontweight='bold')

# Barvy pro silnici a trail
colors = {'road': 'steelblue', 'trail': 'darkorange'}

# === Graf 1 — absolutní podíl pomalých ===
x = np.arange(len(exp_order))
width = 0.35

bars_road = ax1.bar(x - width/2,
                    [pivot.loc[exp, 'road'] for exp in exp_order],
                    width, label='Silnice (Road)',
                    color=colors['road'], alpha=0.85, edgecolor='white')
bars_trail = ax1.bar(x + width/2,
                     [pivot.loc[exp, 'trail'] for exp in exp_order],
                     width, label='Trail',
                     color=colors['trail'], alpha=0.85, edgecolor='white')

# Referenční linie — průměrný podíl pomalých (~33%)
ax1.axhline(33.3, color='black', linewidth=1, linestyle='--', alpha=0.5,
            label='Průměr (33%)')

for bars in [bars_road, bars_trail]:
    for bar in bars:
        h = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2, h + 0.3,
                 f'{h:.1f}%', ha='center', va='bottom', fontsize=9)

ax1.set_xlabel('Zkušenost závodníka', fontsize=11)
ax1.set_ylabel('Podíl závodníků "pomalý" (%)', fontsize=11)
ax1.set_title('Podíl pomalých běžců dle zkušeností', fontsize=11)
ax1.set_xticks(x)
ax1.set_xticklabels([exp.capitalize() for exp in exp_order], fontsize=10)
ax1.legend(title='Povrch', fontsize=9)
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0, 50)

# === Graf 2 — relativní handicap nováčků oproti veteránům ===
bars_rel = ax2.bar(['Silnice (Road)', 'Trail'],
                   [rel_disadv['road'], rel_disadv['trail']],
                   color=[colors['road'], colors['trail']],
                   alpha=0.85, edgecolor='white', width=0.5)

for bar in bars_rel:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, h + 0.2,
             f'+{h:.2f} pb', ha='center', va='bottom',
             fontsize=11, fontweight='bold')

ax2.set_xlabel('Povrch závodu', fontsize=11)
ax2.set_ylabel('Relativní nevýhoda (procentní body)', fontsize=11)
ax2.set_title('Handicap nováčků oproti veteránům\n(rozdíl v podílu pomalých)',
              fontsize=11)
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim(0, max(rel_disadv.max() * 1.25, 15))

plt.tight_layout()
plt.savefig(DATA_DIR / '02_novacci_road_vs_trail.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graf uložen do data/processed/02_novacci_road_vs_trail.png")

### 📊 Interpretace grafů

**Graf vlevo — Podíl pomalých běžců dle zkušeností:**

Trend od nováčků k veteránům jasně potvrzuje, že s rostoucí zkušeností klesá
podíl pomalých běžců — to platí na obou površích.

- **Silnice (road):** podíl pomalých klesá z ~42 % (nováčci) na ~30 % (veteráni)
- **Trail:** podíl pomalých klesá z ~40 % (nováčci) na ~31 % (veteráni)

Nováčci jsou na obou površích výrazně častěji v nejpomalejší třetině startovního
pole než zkušení běžci a veteráni.

**Graf vpravo — Relativní handicap nováčků oproti veteránům:**

Tento graf přináší překvapivé zjištění, které **vyvrací původní hypotézu**:

- Předpoklad: trail penalizuje nezkušenost více kvůli technické náročnosti.
- Data: handicap nováčků je **větší na silnici (~+12 pb) než na trailu (~+9 pb)**.

**Proč tomu tak je?**

1. **Self-selection bias (bariéra vstupu):** Na silniční ultramaraton se může
   přihlásit prakticky kdokoliv — variabilita kondice nováčků je vysoká.
   Naopak na trail jdou již často připravenější běžci (mají naběhané silniční
   maratony nebo trailové tréninky, které dataset nevidí jako 'předchozí start').

2. **Terénní limit rychlosti:** Technický terén (kameny, bláto, prudké výstupy)
   přirozeně omezuje maximální rychlost všech běžců — i veteráni musí zpomalit.
   Silnice naopak přeje plynulému běhu, kde dokonalý pacing a vytrvalost
   (kde mají veteráni převahu) vytváří větší rozdíly.

**Závěr:** Hypotéza se nepotvrdila. Silniční ultramaratony rozevírají rozdíl
mezi začátečníky a zkušenými více než trailové.

## 5. Statistická validace (chi-square test)

In [ ]:
print("Chi-square testy nezávislosti: závisí speed_cat na experience_cat?\n")

chi2_results = {}
for surface in surface_order:
    subset = df_filtered[df_filtered['surface'] == surface]
    ct = pd.crosstab(subset['experience_cat'], subset['speed_cat'])
    chi2, p, dof, _ = chi2_contingency(ct)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    chi2_results[surface] = {'chi2': chi2, 'p': p, 'sig': sig, 'n': len(subset)}
    print(f"  {surface.upper():5s}: chi2={chi2:8.1f}, p={p:.2e} {sig:3s}  (n={len(subset):,})")

print("\nLegenda: *** p<0.001  ** p<0.01  * p<0.05  ns = nesignifikantní")
print("\nZávěr: Rozdíly v rozložení speed_cat dle zkušenosti jsou na obou")
print("površích statisticky vysoce signifikantní — pozorované vzorce nejsou náhodné.")

## 6. Souhrn a business interpretace

In [ ]:
print("=" * 60)
print("SOUHRN — Nováčci na road vs. trail závodech")
print("=" * 60)
print()
print(f"Celkem analyzováno: {len(df_filtered):,} závodníků")
print(f"  → road:  {len(df_filtered[df_filtered['surface']=='road']):,}")
print(f"  → trail: {len(df_filtered[df_filtered['surface']=='trail']):,}")
print()
print("NÁLEZ:")
print()
print("  Podíl pomalých běžců (speed_cat = pomalý):")
for surf in surface_order:
    nov = pivot.loc['nováček', surf]
    vet = pivot.loc['veterán', surf]
    diff = nov - vet
    print(f"    {surf.capitalize():6s}: Nováček {nov:.1f}%  vs.  Veterán {vet:.1f}%   → rozdíl +{diff:.2f} pb")
print()

# Dynamické rozhodnutí — kde je handicap větší
worse_surface = rel_disadv.idxmax()
better_surface = rel_disadv.idxmin()
print(f"  → Větší handicap nováčků na: {worse_surface.upper()} (+{rel_disadv[worse_surface]:.2f} pb)")
print(f"  → Menší handicap nováčků na: {better_surface.upper()} (+{rel_disadv[better_surface]:.2f} pb)")
print()

if worse_surface == 'trail':
    print("INTERPRETACE: Hypotéza POTVRZENA.")
    print("  Trail penalizuje nezkušenost více než silnice.")
else:
    print("INTERPRETACE: Hypotéza VYVRÁCENA.")
    print("  Handicap nováčků je překvapivě VĚTŠÍ na silnici než na trailu.")
    print("  Příčiny:")
    print("    1. Self-selection bias — na trail jdou již připravenější běžci")
    print("    2. Terénní limit rychlosti — technický terén komprimuje rozdíly")
print()
print("BUSINESS DOPORUČENÍ:")
print("  Silniční ultramaratony:")
print("    → Nováčci čelí výraznému zpomalení / odpadu (40%+ pomalých)")
print("    → Zavést 'beginner waves', pacery, edukační kampaně o pacing")
print("  Trailové ultramaratony:")
print("    → Handicap menší, ale stále ~40 % nováčků končí jako pomalí")
print("    → Nabízet kratší varianty (20-30 km) jako 'onboarding' nováčků")
print("  Pro Power BI dashboard:")
print("    → Segmentace experience_cat × surface jako klíčová dimenze")
print("    → KPI: % nováčků v 'pomalý' segmentu (proxy pro retention risk)")

## Shrnutí

**Metoda:** Pivot analýza podílu pomalých běžců (`speed_cat == 'pomalý'`) dle
zkušenostní kategorie × povrch. Relativní handicap vyjádřen jako rozdíl mezi
nováčky a veterány v procentních bodech. Statistická validace chi-square testem
nezávislosti pro každý povrch zvlášť.

**Data:** ~920 tisíc závodníků na road a trail ultramaratonech s validním
rozřazením zkušeností a rychlosti. `speed_cat` je počítán per event (z notebooku 00).

**Klíčový nález:** Hypotéza o vyšší penalizaci nezkušenosti na trailu byla
**vyvrácena**. Relativní handicap nováčků vůči veteránům je vyšší na silničních
závodech (~+12 pb) než na trailu (~+9 pb). Pravděpodobné příčiny: self-selection
bias (na trail jdou připravenější běžci) a fyzikální limit rychlosti v náročném
terénu, který komprimuje rozdíly mezi skupinami.

**Limitace:**
- `experience_cat` měří jen počet startů v ultramaratonském datasetu — neviditelné
  jsou silniční maratony a trail tréninky, což zesiluje self-selection bias
- 'Nováček' na trailu může být reálně zkušený běžec přicházející z jiné disciplíny
- `speed_cat` per event srovnává relativní pozici, ne absolutní rychlost
- Analýza nezohledňuje vliv počasí, věku a pohlaví (řešeno v jiných úlohách)

**Účel:** Podklad pro Power BI dashboard — segmentační dimenze pro analýzu
retention rizika nováčků dle typu závodu.

**Další notebook:** `03_4ft_uloha1.ipynb` — 4ft-Miner úloha (DM část projektu)